# Lábia — o modelo ajustado na CPU (a demo ao vivo)

A demo no Hugging Face roda numa CPU gratuita de 2 núcleos. Este notebook, também só com CPU (não gasta a cota de GPU), responde três perguntas antes de publicar:

1. o GGUF do modelo ajustado funciona com o cliente da demo (`labia.local_llm`), chamando ferramentas de verdade?
2. quanto tempo leva cada resposta com 2 núcleos (como no Space) e com 4?
3. a versão de 4 bits (q4_K_M) vale a troca: quanto menor e mais rápida ela fica?

As poucas conversas daqui usam um cliente roteirizado e servem de **teste de fumaça**. Os vereditos delas não são métrica.

In [ ]:
import ctypes, json, os, statistics, subprocess, sys, time
from pathlib import Path
LABIA_COMMIT = "c9aa1f754ef5e60265f9823c8d535d800f6ce0c2"
OUT = Path("/kaggle/working/cpu"); OUT.mkdir(parents=True, exist_ok=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cpu",
                    "llama-cpp-python==0.3.36", f"git+https://github.com/deniscarvalho-tech/labia.git@{LABIA_COMMIT}"],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-500:])
print("núcleos:", os.cpu_count())

## 1. O GGUF do treino e a versão de 4 bits

A quantização para 4 bits (q4_K_M) parte do GGUF de 8 bits, pela própria biblioteca do llama.cpp. O ideal seria partir dos pesos em 16 bits; a diferença de qualidade fica registrada como limitação da demo, não da avaliação (que usa o de 8 bits).

In [ ]:
import llama_cpp
q8 = next(Path("/kaggle/input").rglob("labia-sft-q8_0.gguf"))
q4 = Path("/kaggle/working/labia-sft-q4_k_m.gguf")  # vai para a demo
params = llama_cpp.llama_model_quantize_default_params()
params.ftype = llama_cpp.LLAMA_FTYPE_MOSTLY_Q4_K_M
params.allow_requantize = True
params.nthread = os.cpu_count()
start = time.perf_counter()
rc = llama_cpp.llama_model_quantize(str(q8).encode(), str(q4).encode(), ctypes.byref(params))
assert rc == 0, f"a quantização falhou ({rc})"
sizes = {"q8_0": round(q8.stat().st_size / 1024**3, 2), "q4_k_m": round(q4.stat().st_size / 1024**3, 2)}
print(sizes, f"quantização em {time.perf_counter() - start:.0f} s")

## 2. Conversas de fumaça e latência

Para cada versão e número de núcleos: um golpe e um cliente honesto do teste, até 3 turnos. A primeira resposta paga o prompt inteiro (política e ferramentas, cerca de 3 mil tokens); as seguintes reaproveitam o que já foi lido.

In [ ]:
from labia.agents import LLMAgent, ScriptedCustomer
from labia.conversation import Conversation
from labia.local_llm import LlamaCppClient
from labia.scenarios.generator import generate

test = sorted(generate("test"), key=lambda s: s.id)
picks = [next(s for s in test if s.situation == "D2-nao-entregue" and not s.legit and s.tactic.value == "institutional_threat"),
         next(s for s in test if s.situation == "B1-estorno-tarifa" and s.legit and s.tactic.value == "none")]
runs = []
for variant, path in [("q4_k_m", q4), ("q8_0", q8)]:
    for threads in (2, 4):
        client = LlamaCppClient(str(path), "Qwen/Qwen3-4B-Instruct-2507", n_ctx=8192, n_threads=threads)
        for s in picks:
            conv = Conversation(s, LLMAgent(client, variant, temperature=0.7, max_tokens=512, seed=0), max_turns=3)
            customer = ScriptedCustomer(s)
            start = time.perf_counter()
            while not conv.done:
                conv.customer_says(customer.next_message(conv.dialogue))
            rec = conv.record(model_id=f"sft-{variant}")
            calls = [{"turno": st.turn, "latencia_s": round(st.latency_s, 1), "tokens_prompt": st.prompt_tokens,
                      "tokens_saida": st.completion_tokens} for st in rec.agent_stats]
            runs.append({"versao": variant, "nucleos": threads, "cenario": s.id, "veredito_fumaca": rec.verdict.value,
                         "ferramentas": [c["name"] for c in rec.calls], "chamadas_validas": f"{rec.n_valid_calls}/{rec.n_calls}",
                         "segundos_conversa": round(time.perf_counter() - start, 1), "chamadas_ao_modelo": calls})
            print(json.dumps(runs[-1], ensure_ascii=False)[:600])
        del client

## 3. O estado pré-aquecido (o que a demo faz ao ligar)

A demo lê uma vez a política e as ferramentas de cada cenário jogável e guarda o estado do modelo. Aqui medimos a primeira resposta depois de restaurar esse estado, contra a primeira resposta a frio medida acima.

In [ ]:
from labia.contracts.common import Defense
from labia.prompts import agent_system_prompt
from labia.sandbox import get_sector

client = LlamaCppClient(str(q4), "Qwen/Qwen3-4B-Instruct-2507", n_ctx=8192, n_threads=2)
warm_runs = []
for s in picks:
    start = time.perf_counter()
    client.warm(s.id, {"role": "system", "content": agent_system_prompt(s, Defense.STANDARD)}, get_sector(s.sector).tool_schemas())
    warm_s = time.perf_counter() - start
    client.restore(s.id)
    conv = Conversation(s, LLMAgent(client, "q4_k_m", temperature=0.7, max_tokens=512, seed=0), max_turns=1)
    conv.customer_says(ScriptedCustomer(s).next_message([]))
    rec = conv.record(model_id="sft-q4_k_m")
    warm_runs.append({"cenario": s.id, "aquecimento_s": round(warm_s, 1),
                      "primeira_chamada_depois_de_restaurar_s": round(rec.agent_stats[0].latency_s, 1),
                      "chamadas_validas": f"{rec.n_valid_calls}/{rec.n_calls}"})
    print(warm_runs[-1])

## 4. Resumo

In [ ]:
summary = {"nucleos_da_maquina": os.cpu_count(), "tamanho_gb": sizes, "por_configuracao": {}, "pre_aquecido_q4_2nucleos": warm_runs}
for variant in ("q4_k_m", "q8_0"):
    for threads in (2, 4):
        rs = [r for r in runs if r["versao"] == variant and r["nucleos"] == threads]
        first = [r["chamadas_ao_modelo"][0]["latencia_s"] for r in rs if r["chamadas_ao_modelo"]]
        later = [c["latencia_s"] for r in rs for c in r["chamadas_ao_modelo"][1:]]
        summary["por_configuracao"][f"{variant}-{threads}nucleos"] = {
            "primeira_chamada_s_mediana": statistics.median(first) if first else None,
            "chamadas_seguintes_s_mediana": statistics.median(later) if later else None,
            "conversas": len(rs),
        }
(OUT / "resumo.json").write_text(json.dumps({"resumo": summary, "conversas": runs}, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))